## 2. Transformation et stockage temporaire des données


**Objectif**

Stocker les fichiers csv générés des API vers la base de données en tant que tables brutes sans aucune transformation. Cette étape permet d'avoir les données brutes dans la base en cas de problème on peut les restaurer pour une utilisation.

In [1]:
# %pip install dotenv

In [32]:
import os
import pandas as pd
# import geopandas as gpd
from sqlalchemy import create_engine, text
from dotenv import load_dotenv

# 1. Chargement du fichier .env pour récupérer tes identifiants secrets
load_dotenv()

db_user = os.getenv("DB_USER")
db_password = os.getenv("DB_PASSWORD")
db_name = "retail_db"

# 2. DETECTION DE L'ENVIRONNEMENT (Le correctif pour ton erreur)
# Si on est dans Docker, un dossier '/opt/airflow' existe. Sinon, on est sur ton Windows.
if os.path.exists("/opt/airflow"):
    db_host = "postgres-gis"  # Connexion interne Docker
    print("[Info] Exécution détectée dans le conteneur DOCKER.")
else:
    db_host = "localhost"     # Connexion depuis ton VS Code Windows
    print("[Info] Exécution détectée en LOCAL (Windows VS Code).")

# 3. Construction de la chaîne de connexion
DATABASE_URL = f"postgresql://{db_user}:{db_password}@{db_host}:5432/{db_name}"
engine = create_engine(DATABASE_URL)

# 4. Test de la connexion
with engine.connect() as conn:
    conn.execute(text("CREATE EXTENSION IF NOT EXISTS postgis;"))
    # conn.commit()

print("[OK] Connexion sécurisée établie avec succès ! extension PostGIS active.")

python-dotenv could not parse statement starting at line 20
python-dotenv could not parse statement starting at line 21


[Info] Exécution détectée en LOCAL (Windows VS Code).
[OK] Connexion sécurisée établie avec succès ! extension PostGIS active.


Chargement, Nettoyage et Spatialisation des données
C'est ici qu'on ouvre les fichiers CSV générés par le premier notebook (qui se trouvent dans /opt/airflow/donnees/) pour les pousser proprement dans les tables SQL en convertissant les coordonnées GPS en vrais objets géométriques.

In [16]:
import os
import pandas as pd

# 1. Détection dynamique du dossier selon l'environnement
if os.path.exists("/opt/airflow"):
    # On est dans Docker
    DOSSIER_REEL = "/opt/airflow/donnees/"
else:
    # On est sur ton Windows (VS Code local)
    DOSSIER_REEL = "./" # Signifie "le dossier actuel où se trouve le notebook"

# 2. Construction du chemin du fichier
chemin_population = os.path.join(DOSSIER_REEL, "population_pouvoir_achat.csv")

# 3. Vérification de sécurité avant la lecture
if not os.path.exists(chemin_population):
    # Optionnel : Si tu as nommé ton fichier "population_attractivité.csv" à l'étape d'avant, modifie la chaîne au-dessus
    print(f"[Contenu du dossier] : {os.listdir(DOSSIER_REEL)}") # Te montre ce qui est dispo dans le dossier
    raise FileNotFoundError(f"[ERREUR] Le fichier est introuvable à cet endroit : {chemin_population}")

# 4. Lecture sécurisée
df_pop = pd.read_csv(chemin_population)
print(f"[✓] Fichier chargé avec succès ! Nombre de lignes : {len(df_pop)}")

# Stockage de la table temporaire dans la base de données PostgreSQL
df_pop.to_sql('stg_population_pouvoir_achat', con=engine, if_exists='replace', index=False)
df_pop

[✓] Fichier chargé avec succès ! Nombre de lignes : 1795


,GEO,TIME_PERIOD,FILOSOFI_MEASURE,OBS_VALUE_NIVEAU
0,2026-COM-75056,2023,MED_SL,33650.0
1,2026-COM-75056,2023,PR_MD60,16.8
2,2026-COM-77001,2023,MED_SL,34840.0
3,2026-COM-77002,2023,MED_SL,29130.0
4,2026-COM-77003,2023,MED_SL,30310.0
...,...,...,...,...
1790,2026-COM-95678,2023,MED_SL,34300.0
1791,2026-COM-95680,2023,MED_SL,17750.0
1792,2026-COM-95680,2023,PR_MD60,39.4
1793,2026-COM-95682,2023,MED_SL,28800.0


In [17]:
#  Construction du chemin du fichier
chemin_population1 = os.path.join(DOSSIER_REEL, "population_attractivité.csv")
df_pop1 = pd.read_csv(chemin_population1)

#  Vérification de sécurité avant la lecture
if not os.path.exists(chemin_population1):
    print(f"[Contenu du dossier] : {os.listdir(DOSSIER_REEL)}") # Te montre ce qui est dispo dans le dossier
    raise FileNotFoundError(f"[ERREUR] Le fichier est introuvable à cet endroit : {chemin_population1}")

# Stockage de la table temporaire dans la base de données PostgreSQL
df_pop1.to_sql('stg_population_attractivité', con=engine, if_exists='replace', index=False)

1000

In [18]:
#  Construction du chemin du fichier
chemin_population2 = os.path.join(DOSSIER_REEL, "population_emploi.csv")
df_pop2 = pd.read_csv(chemin_population2)

#  Vérification de sécurité avant la lecture
if not os.path.exists(chemin_population2):
    print(f"[Contenu du dossier] : {os.listdir(DOSSIER_REEL)}") # Te montre ce qui est dispo dans le dossier
    raise FileNotFoundError(f"[ERREUR] Le fichier est introuvable à cet endroit : {chemin_population2}")

# Stockage de la table temporaire dans la base de données PostgreSQL
df_pop2.to_sql('stg_population_emploi', con=engine, if_exists='replace', index=False)

532

In [29]:
#  Construction du chemin du fichier
chemin_population3 = os.path.join(DOSSIER_REEL, "population_infrastructure.csv")
df_pop3 = pd.read_csv(chemin_population3)

#  Vérification de sécurité avant la lecture
if not os.path.exists(chemin_population3):
    print(f"[Contenu du dossier] : {os.listdir(DOSSIER_REEL)}") # Te montre ce qui est dispo dans le dossier
    raise FileNotFoundError(f"[ERREUR] Le fichier est introuvable à cet endroit : {chemin_population3}")

# Stockage de la table temporaire dans la base de données PostgreSQL
df_pop3.to_sql('stg_population_infrastructure', con=engine, if_exists='replace', index=False)
df_pop3

,GEO,BPE_MEASURE,FACILITY_SDOM,FACILITY_TYPE,TIME_PERIOD,UNIT_MEASURE,FACILITY_DOM,OBS_STATUS,UNIT_MULT,OBS_VALUE_NIVEAU
0,2026-COM-75056,FACILITIES,E1,E107,2025,NR,E,A,0,7.0
1,2026-COM-77111,FACILITIES,E1,E107,2025,NR,E,A,0,1.0
2,2026-COM-91377,FACILITIES,E1,E107,2025,NR,E,A,0,1.0
3,2026-COM-93073,FACILITIES,E1,E107,2025,NR,E,A,0,1.0
4,2026-COM-75056,FACILITIES,E1,E108,2025,NR,E,A,0,27.0
...,...,...,...,...,...,...,...,...,...,...
11310,2026-COM-95424,FACILITIES,D3,D307,2025,NR,D,A,0,5.0
11311,2026-COM-95369,FACILITIES,D3,D307,2025,NR,D,A,0,1.0
11312,2026-COM-95582,FACILITIES,D3,D307,2025,NR,D,A,0,9.0
11313,2026-COM-95539,FACILITIES,D3,D307,2025,NR,D,A,0,4.0


In [26]:
#  Construction du chemin du fichier
chemin_population4 = os.path.join(DOSSIER_REEL, "population_pouvoir_achat.csv")
df_pop4 = pd.read_csv(chemin_population4)

#  Vérification de sécurité avant la lecture
if not os.path.exists(chemin_population4):
    print(f"[Contenu du dossier] : {os.listdir(DOSSIER_REEL)}") # Te montre ce qui est dispo dans le dossier
    raise FileNotFoundError(f"[ERREUR] Le fichier est introuvable à cet endroit : {chemin_population4}")

# Stockage de la table temporaire dans la base de données PostgreSQL
df_pop4.to_sql('stg_population_pouvoir_achat', con=engine, if_exists='replace', index=False)

795

In [31]:
#  Construction du chemin du fichier
chemin_population5 = os.path.join(DOSSIER_REEL, "population_logement.csv")
df_pop5 = pd.read_csv(chemin_population5)

#  Vérification de sécurité avant la lecture
if not os.path.exists(chemin_population5):
    print(f"[Contenu du dossier] : {os.listdir(DOSSIER_REEL)}") # Te montre ce qui est dispo dans le dossier
    raise FileNotFoundError(f"[ERREUR] Le fichier est introuvable à cet endroit : {chemin_population5}")

# Stockage de la table temporaire dans la base de données PostgreSQL
df_pop5.to_sql('stg_population_logement', con=engine, if_exists='replace', index=False)
df_pop5

,GEO,TIME_PERIOD,NOR,OCS,L_STAY,OBS_VALUE_NIVEAU
0,2026-COM-75056,2023,_T,DW_MAIN,Y10T19,198821.63183
1,2026-COM-75056,2023,_T,DW_MAIN,Y20T29,138351.28940
2,2026-COM-75056,2023,_T,DW_MAIN,Y2T4,271313.38030
3,2026-COM-75056,2023,_T,DW_MAIN,Y5T9,196659.40251
4,2026-COM-75056,2023,_T,DW_MAIN,Y_GE30,148080.43683
...,...,...,...,...,...,...
27847,2026-COM-95690,2017,R1,DW_MAIN,_T,1.01383
27848,2026-COM-95690,2017,R2,DW_MAIN,_T,6.07222
27849,2026-COM-95690,2017,R3,DW_MAIN,_T,17.11712
27850,2026-COM-95690,2017,R4,DW_MAIN,_T,25.97596


In [33]:
#  Construction du chemin du fichier
chemin_population6 = os.path.join(DOSSIER_REEL, "population_recensement.csv")
df_pop6 = pd.read_csv(chemin_population6)

#  Vérification de sécurité avant la lecture
if not os.path.exists(chemin_population6):
    print(f"[Contenu du dossier] : {os.listdir(DOSSIER_REEL)}") # Te montre ce qui est dispo dans le dossier
    raise FileNotFoundError(f"[ERREUR] Le fichier est introuvable à cet endroit : {chemin_population6}")

# Stockage de la table temporaire dans la base de données PostgreSQL
df_pop6.to_sql('stg_population_recensement', con=engine, if_exists='replace', index=False)

384

In [ ]:
df_pop6

In [23]:
#  Construction du chemin du fichier
chemin_population7 = os.path.join(DOSSIER_REEL, "population_reference.csv")
df_pop7 = pd.read_csv(chemin_population7)

#  Vérification de sécurité avant la lecture
if not os.path.exists(chemin_population7):
    print(f"[Contenu du dossier] : {os.listdir(DOSSIER_REEL)}") # Te montre ce qui est dispo dans le dossier
    raise FileNotFoundError(f"[ERREUR] Le fichier est introuvable à cet endroit : {chemin_population7}")

# Stockage de la table temporaire dans la base de données PostgreSQL
df_pop7.to_sql('stg_population_reference', con=engine, if_exists='replace', index=False)

798

In [24]:
#  Construction du chemin du fichier
chemin_population8 = os.path.join(DOSSIER_REEL, "population_remuneration.csv")
df_pop8 = pd.read_csv(chemin_population8)

#  Vérification de sécurité avant la lecture
if not os.path.exists(chemin_population8):
    print(f"[Contenu du dossier] : {os.listdir(DOSSIER_REEL)}") # Te montre ce qui est dispo dans le dossier
    raise FileNotFoundError(f"[ERREUR] Le fichier est introuvable à cet endroit : {chemin_population8}")

# Stockage de la table temporaire dans la base de données PostgreSQL
df_pop8.to_sql('stg_population_remuneration', con=engine, if_exists='replace', index=False)

596

In [25]:
#  Construction du chemin du fichier
chemin_population9 = os.path.join(DOSSIER_REEL, "population_transport.csv")
df_pop9 = pd.read_csv(chemin_population9)

#  Vérification de sécurité avant la lecture
if not os.path.exists(chemin_population9):
    print(f"[Contenu du dossier] : {os.listdir(DOSSIER_REEL)}") # Te montre ce qui est dispo dans le dossier
    raise FileNotFoundError(f"[ERREUR] Le fichier est introuvable à cet endroit : {chemin_population9}")

# Stockage de la table temporaire dans la base de données PostgreSQL
df_pop9.to_sql('stg_population_transport', con=engine, if_exists='replace', index=False)

192

In [14]:
#  Construction du chemin du fichier
chemin_commerce = os.path.join(DOSSIER_REEL, "commerces_idf.pkl")
df_commerce = pd.read_pickle(chemin_commerce)

#  Vérification de sécurité avant la lecture
if not os.path.exists(chemin_commerce):
    print(f"[Contenu du dossier] : {os.listdir(DOSSIER_REEL)}") # Te montre ce qui est dispo dans le dossier
    raise FileNotFoundError(f"[ERREUR] Le fichier est introuvable à cet endroit : {chemin_commerce}")

# Stockage de la table temporaire dans la base de données PostgreSQL
df_commerce.to_sql('stg_commerces_idf', con=engine, if_exists='replace', index=False)

c:\Users\jessi\Desktop\Mes_cours_Ynov\projet_fin_etude\BLOC1\.venv\Lib\site-packages\pandas\io\sql.py:2078: SAWarning: Did not recognize type 'geometry' of column 'coordonnee_geographique'
  self.meta.reflect(


324

In [14]:
#  Construction du chemin du fichier
chemin_velo = os.path.join(DOSSIER_REEL, "comptage-velo-compteurs.csv")
df_velo = pd.read_csv(chemin_velo, sep=";")

#  Vérification de sécurité avant la lecture
if not os.path.exists(chemin_velo):
    print(f"[Contenu du dossier] : {os.listdir(DOSSIER_REEL)}") # Te montre ce qui est dispo dans le dossier
    raise FileNotFoundError(f"[ERREUR] Le fichier est introuvable à cet endroit : {chemin_velo}")



In [15]:
# Stockage de la table temporaire dans la base de données PostgreSQL
df_velo.to_sql('stg_velos_comptage', con=engine, if_exists='replace', index=False)

c:\Users\jessi\Desktop\Mes_cours_Ynov\projet_fin_etude\BLOC1\.venv\Lib\site-packages\pandas\io\sql.py:2078: SAWarning: Did not recognize type 'geometry' of column 'geom'
  self.meta.reflect(


113

In [16]:
df_velo.head()

,Identifiant du compteur,Nom du compteur,Identifiant du site de comptage,Nom du site de comptage,Identifiant du channel,Nom du channel,Date d'installation du site de comptage,Lien vers photo du site de comptage,Identifiant technique compteur,Coordonnées géographiques,ID Photos,test_lien_vers_photos_du_site_de_comptage_,id_photo_1,url_sites,type_dimage
0,100003096-353242251,97 avenue Denfert Rochereau SO-NE,100003096,97 avenue Denfert Rochereau,353242251,SO-NE,2012-02-21,['https://filer.eco-counter-tools.com/file/10/...,Y2111121725,"48.83504, 2.33314",['https://filer.eco-counter-tools.com/file/10/...,['https://filer.eco-counter-tools.com/file/10/...,['https:,https://www.eco-visio.net/Photos/100003096,jpg']
1,100003098-101003098,106 avenue Denfert Rochereau NE-SO,100003098,106 avenue Denfert Rochereau,101003098,NE-SO,2012-02-21,['https://filer.eco-counter-tools.com/file/ad/...,Y2H16029278,"48.83507, 2.33305",['https://filer.eco-counter-tools.com/file/ad/...,['https://filer.eco-counter-tools.com/file/ad/...,['https:,https://www.eco-visio.net/Photos/100003098,jpg']
2,100006300-101006300,135 avenue Daumesnil SE-NO,100006300,135 avenue Daumesnil,101006300,SE-NO,2013-01-18,['https://filer.eco-counter-tools.com/file/97/...,X2H18086316,"48.843435, 2.383378",['https://filer.eco-counter-tools.com/file/97/...,['https://filer.eco-counter-tools.com/file/97/...,['https:,https://www.eco-visio.net/Photos/100006300,jpg']
3,100007049-101007049,28 boulevard Diderot O-E,100007049,28 boulevard Diderot,101007049,O-E,2013-01-17,['https://filer.eco-counter-tools.com/file/9b/...,Y2H15027244,"48.84613, 2.37559",['https://filer.eco-counter-tools.com/file/9b/...,['https://filer.eco-counter-tools.com/file/9b/...,['https:,https://www.eco-visio.net/Photos/100007049,jpg']
4,100007049-102007049,28 boulevard Diderot E-O,100007049,28 boulevard Diderot,102007049,E-O,2013-01-17,['https://filer.eco-counter-tools.com/file/9b/...,Y2H15027244,"48.84613, 2.37559",['https://filer.eco-counter-tools.com/file/9b/...,['https://filer.eco-counter-tools.com/file/9b/...,['https:,https://www.eco-visio.net/Photos/100007049,jpg']
